# Système de recommandation hybride : réduire le biais de popularité (Goodreads)

**Équipe :** Amine et Ella · **Owner :** les deux (Amine assemble les sections 1, 3, 4, 5 ; Ella les sections 2 et 6).

**Ce que fait ce notebook :** il suit les étapes de l'énoncé (1 à 6) et affiche les tableaux, figures et
exemples utilisés dans le rapport. La logique est dans `src/` (code commenté) ; le notebook l'appelle.

**Entrée :** les fichiers produits par le pipeline (à lancer avant, dans cet ordre) :

```
python scripts/run_pipeline.py --force     # données, modèles, évaluation
python scripts/tune_k.py                   # choix de k sur la validation
python scripts/run_pipeline.py             # réévaluation du test avec ce k
```

**Sortie :** les chiffres et figures du rapport (aucun réentraînement ici).

In [ ]:
import json, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import Image, display

from src.utils.config import load_config
from src.labels import METRIC_LABEL, MODEL_LABEL, fr_metric, model_label, popularity_thresholds

cfg = load_config()
P, T, F = cfg["paths"]["processed_dir"], cfg["paths"]["tables_dir"], cfg["paths"]["figures_dir"]
pd.set_option("display.max_columns", 30)

def show(path, width=850):
    if Path(path).exists():
        display(Image(filename=str(path), width=width))
    else:
        print("Figure absente :", path, "-> lancer le pipeline.")

## 1. Domaine et jeu de données

Domaine : **livres** (bandes dessinées et romans graphiques). Jeu de données public **UCSD Book Graph**
(Goodreads, Wan & McAuley, 2017), sous-ensemble *Comics & Graphic* : 89 411 livres et 7 347 630
interactions. Il contient les **interactions utilisateur-livre** (lu, note, date) et des **descriptions**
des livres (texte + étagères populaires des lecteurs), comme demandé.

Pour tenir sur un ordinateur portable, nous échantillonnons des lecteurs (`data.n_users_sample`).

In [ ]:
print("Sous-ensemble :", cfg["data"]["interactions_file"], "|", cfg["data"]["books_file"])
print("Lecteurs échantillonnés :", cfg["data"]["n_users_sample"],
      "| interaction positive : lu ou note >=", cfg["data"]["positive_rating"])

## 2. Analyse et préparation des données

Statistiques à chaque étape (brut, nettoyé, entraînement) : nombre de lecteurs, de livres,
d'interactions, **taux de remplissage** (densité) et **sparsité** de la matrice, concentration de la
popularité (indice de Gini). Nettoyage : interactions positives uniquement, filtre *k-core* (au moins
5 interactions par lecteur et par livre), textes nettoyés. Découpage **temporel par lecteur** :
les 20 % d'interactions les plus récentes en test, les 10 % précédentes en validation.

In [ ]:
stats = pd.read_csv(T / "dataset_stats.csv")
stats

In [ ]:
n_items = np.load(P / "n_items.npy")
mid_min, head_min = popularity_thresholds(n_items)
print(f"Livres : {len(n_items)} | médiane des lecteurs par livre : {np.median(n_items):.0f} | max : {n_items.max():.0f}")
print(f"Best-seller : >= {head_min:.0f} lecteurs (top 5 %) | intermédiaire : >= {mid_min:.0f} (top 20 %) | pépite : le reste (80 %)")
top20 = np.sort(n_items)[::-1][: int(0.2 * len(n_items))].sum() / n_items.sum()
print(f"Les 20 % de livres les plus lus concentrent {top20:.0%} des interactions d'entraînement.")
show(F / "long_tail.png")

In [ ]:
cold_books = json.load(open(P / "cold_books.json", encoding="utf-8"))["book_ids"]
cold_users = json.load(open(P / "cold_users.json", encoding="utf-8"))["user_ids"]
print(f"Démarrage à froid simulé : {len(cold_books)} livres et {len(cold_users)} lecteurs rendus « nouveaux »")
print("(les données datent de 2017 : on cache la plupart des interactions d'entraînement de ces livres / lecteurs)")

## 3. Approches de base

- **Popularité** (baseline) : recommande les livres les plus lus.
- **ALS** (filtrage collaboratif, feedback implicite) : un vecteur par lecteur et par livre, appris sur
  la matrice lecteurs × livres avec une confiance `1 + α·r` (`src/models/als_model.py`).
- **TF-IDF** (basé sur le contenu) : un vecteur par livre à partir de sa description et de ses
  étagères (sans les étagères de rangement comme « to-read ») ; le profil d'un lecteur est la moyenne des
  livres qu'il a aimés (`src/models/content_tfidf.py`).
- *Approche récente (perspective) : embeddings de phrases MiniLM — non implémentée.*

In [ ]:
from src import recommend
art = recommend.load_artifacts(cfg)
print("Matrice d'entraînement :", art["train"].shape, "| interactions :", art["train"].nnz)
print("Facteurs ALS :", art["als"].item_factors.shape, "| vecteurs TF-IDF :", art["item_vectors"].shape)

## 4. Approche hybride : pondérée, avec un poids adaptatif par livre

`score(u, i) = α_i · ALS(u, i) + (1 − α_i) · contenu(u, i)` avec `α_i = n_i / (n_i + k)`,
où `n_i` est le nombre de lecteurs du livre dans l'entraînement (scores normalisés min-max par lecteur).

**Justification :** le vecteur ALS d'un livre n'est fiable que si le livre a assez de lecteurs.
Un livre peu lu est donc classé surtout sur son contenu, un livre très lu surtout par ALS.
`k` = nombre de lecteurs pour lequel les deux branches pèsent autant.

**Choix de k (validation) :** le plus grand k dont le NDCG@20 reste à moins de 5 % du meilleur
(`gate.ndcg_tolerance`) — un compromis explicite entre pertinence et découverte.

In [ ]:
sweep = pd.read_csv(T / "k_sweep_val.csv")
best = json.load(open(T / "best_k.json", encoding="utf-8"))
print("k retenu :", best["k"], "|", best.get("selection_rule", ""))
sweep.round(4)

In [ ]:
show(F / "tradeoff_curve.png")
show(F / "gate_curve.png", width=650)

## 5. Évaluation expérimentale (jeu de test)

L'hybride est comparé à **chaque approche seule** et à la **baseline de popularité**, avec des métriques de
précision (Précision, Rappel, NDCG, HR @K) et **au-delà de la précision** (couverture, nouveauté, part de
pépites). Le **démarrage à froid** est testé à part : livres nouveaux et lecteurs nouveaux.

In [ ]:
test = pd.read_csv(T / "test_metrics.csv", index_col=0)
cols = ["all/precision@20", "all/recall@20", "all/ndcg@20", "all/hr@20",
        "cold_items/recall@20", "cold_users/recall@20", "coverage", "novelty", "tail_share"]
cols = [c for c in cols if c in test.columns]
table = test[cols].copy()
table.index = [model_label(m) for m in table.index]
table.columns = [METRIC_LABEL.get(c, c) for c in cols]
table.round(4)

In [ ]:
show(F / "popularity_of_recs.png")
show(F / "baseline_bars.png")

**Lecture des résultats (à compléter avec les chiffres finaux) :** quel modèle est le plus pertinent,
lequel ouvre le plus la longue traîne, et où l'hybride adaptatif se place. Être honnête : l'hybride
adaptatif fait confiance à ALS pour les livres très lus, et ALS favorise les livres populaires ; son
intérêt principal est sur les **livres nouveaux**, alors que l'hybride 50/50 peut donner plus de diversité
globale.

## 6. Démonstration : recommandations expliquées

Trois lecteurs de démonstration, ALS contre hybride adaptatif, avec l'explication de chaque livre.
L'interface complète : `streamlit run app/streamlit_app.py`.

In [ ]:
pd.set_option("display.max_colwidth", 160)
for user in art["demo_users"][:3]:
    print("=" * 100)
    print("Lecteur", user, "-", art["train"][art["user2idx"][user]].nnz, "livres lus")
    for model in ("als", "gate"):
        recs = recommend.recommend_for_user(user, art, model_name=model, top_n=5)
        print(f"\n  {MODEL_LABEL[model]} :")
        display(recs[["rank", "title", "n", "popularity", "explanation"]])

In [ ]:
# Nouveau lecteur : 3 livres choisis -> recommandations par le contenu seul (alpha = 0)
picked = list(art["books"].sort_values("n_ratings", ascending=False)["book_id"].astype(str).head(3))
print("Livres choisis :", list(art["books"].set_index("book_id").loc[picked, "title"]))
recommend.recommend_for_new_reader(picked, art, top_n=5)[["rank", "title", "n", "popularity", "explanation"]]

## Conclusion

*(À rédiger à partir des chiffres finaux : apport de chaque approche, compromis pertinence / découverte,
résultats en démarrage à froid, limites — données de 2017 et démarrage à froid simulé, un seul genre,
éditions multiples sur Goodreads — et perspectives : poids appris par lecteur, embeddings.)*